🟠 Function F4 — Week 12 | Black-Box Optimisation
### Mike Kennelly | BBO Capstone Project | Final Query

---

## 📌 F4 Function Profile

| Property | Value |
|----------|-------|
| **Dimensions** | 4D |
| **Search space** | [0, 1]^4 |
| **Week** | 12 — FINAL QUERY |
| **All-time best (ATB)** | 0.23759 (Week 2) |
| **W11 result** | 0.23759 — Exact match |
| **Objective** | MAXIMISE ⬆️ |

---

## 🗓️ W12 Decision

**W12 strategy:** ATB OVERRIDE — replicate W2 ATB [0.439249, 0.414994, 0.384687, 0.397917]

**TuRBO direction:** OVERRIDE — non-stationary landscape, only positive result in 11 weeks

---

## 📊 Module 22 Clustering Analysis — W12 Justification

| Property | Value |
|----------|-------|
| **Cluster type** | ISOLATED ATTRACTOR |
| **Cluster detail** | Top-3 spread=0.062. Single narrow positive basin — W2 and W11 (exact repeat) are the only positive results across 11 weeks. 9 surrounding attempts all negative. ATB→centroid dist=0.031. Move any dim >0.05 → fall back to negative. |
| **W12 rationale** | W2 ATB is the only positive result in 11 weeks. W11 exact replication confirmed coords correct (matched 0.23759 exactly). GP sigma=2.25 (non-stationary landscape) — no trustworthy gradient signal to improve on this point. |

---

## 🗺️ Pipeline Overview

```
S0  Config & hyperparameters
S1  Imports
S2  Load data (f4_w12_inputs.npy / f4_w12_outputs.npy)
S3  Historical performance visualisation
S4  Binary labels (top 30%)
S5  CV model comparison (8 classifiers incl. CNN-1D)
S5B CNN inspection (Module 17 learning exercise)
S8  Generate & filter candidates
S9  Gaussian Process regression (Matérn 5/2)
S10 Acquisition functions (EI + UCB)
S11 Per-dimension sensitivity curves
S12 GP acquisition surfaces
S13 Submission dashboard
S14 Final formatted submission (ATB OVERRIDE)
S15 Save hyperparameter record
```


In [ ]:
FUNCTION_ID  = 'F4'
WEEK         = 12
INPUT_FILE   = 'f4_w12_inputs.npy'
OUTPUT_FILE  = 'f4_w12_outputs.npy'

# -- W12 Hyperparameters --------------------------------------
TOP_PERCENTILE    = 30
N_CANDIDATES      = 10000
EXPLOIT_RATIO     = 0.92
EXPLOIT_SIGMA     = 0.012
UCB_KAPPA         = 2.0
EI_XI             = 0.01
GP_RESTARTS       = 10
FILTER_PERCENTILE = 50
RANDOM_SEED       = 42
MAXIMIZE          = True

# Anisotropic sigma (per-dimension from W11 Ollama consensus)
ANISO_SIGMA = [0.012, 0.012, 0.012, 0.012]

NN_SMALL  = (16, 8)
NN_MEDIUM = (64, 32)
NN_LARGE  = (128, 64, 32)

# -- ATB override ---------------------------------------------
ATB_OVERRIDE_STR = '0.439249-0.414994-0.384687-0.397917'
ATB_COORDS       = [float(x) for x in ATB_OVERRIDE_STR.split('-')]

print(f'{FUNCTION_ID} Week {WEEK} -- FINAL QUERY')
print(f'Strategy : ATB OVERRIDE -- replicate W2 ATB [0.439249, 0.414994, 0.384687, 0.397917]')
print(f'ATB      : 0.23759  (Week 2)')
print(f'Override : {ATB_OVERRIDE_STR}')

---
## 📦 STEP 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from scipy.stats import norm
import torch, torch.nn as nn, torch.optim as optim_torch
from sklearn.base import BaseEstimator, ClassifierMixin

np.random.seed(RANDOM_SEED)
print(f"[OK] Imports loaded -- {FUNCTION_ID} Week {WEEK}")

In [ ]:
class TinyCNNClassifier(BaseEstimator, ClassifierMixin):
    """Conv1d(1->8, k=2) -> ReLU -> Dropout(0.3) -> Linear(8->1) -- Module 17 learning exercise."""
    def __init__(self, n_epochs=200, lr=0.01, seed=42):
        self.n_epochs=n_epochs; self.lr=lr; self.seed=seed
    def fit(self, X, y):
        torch.manual_seed(self.seed)
        n_feat=X.shape[1]; self.classes_=np.unique(y)
        class _CNN(nn.Module):
            def __init__(self):
                super().__init__()
                self.conv=nn.Conv1d(1,8,kernel_size=min(2,n_feat),padding=0)
                self.relu=nn.ReLU(); self.drop=nn.Dropout(0.3)
                conv_out=8*max(1,n_feat-min(2,n_feat)+1)
                self.fc=nn.Linear(conv_out,1)
            def forward(self,x):
                x=self.relu(self.conv(x)); x=self.drop(x.view(x.size(0),-1))
                return self.fc(x).squeeze(-1)
        self.model_=_CNN()
        n_pos=y.sum(); n_neg=len(y)-n_pos
        pw=torch.tensor([n_neg/max(n_pos,1)],dtype=torch.float)
        loss_fn=nn.BCEWithLogitsLoss(pos_weight=pw)
        opt=optim_torch.Adam(self.model_.parameters(),lr=self.lr,weight_decay=1e-3)
        Xt=torch.FloatTensor(X).unsqueeze(1); yt=torch.FloatTensor(y.astype(float))
        self.model_.train()
        for _ in range(self.n_epochs):
            opt.zero_grad(); loss_fn(self.model_(Xt),yt).backward(); opt.step()
        return self
    def predict_proba(self,X):
        self.model_.eval()
        with torch.no_grad():
            p=torch.sigmoid(self.model_(torch.FloatTensor(X).unsqueeze(1))).numpy()
        return np.column_stack([1-p,p])
    def predict(self,X): return (self.predict_proba(X)[:,1]>=0.5).astype(int)

---
## 📥 STEP 2 — Load Data

W12 data: `f4_w12_inputs.npy` (4D, includes all W1–W11 evaluations plus W11 result appended).
ATB confirmed at index `np.argmax(y_train)`.


In [ ]:
X_train = np.load(INPUT_FILE)
y_train = np.load(OUTPUT_FILE)
n_samples, n_dims = X_train.shape

best_idx   = np.argmax(y_train)
best_value = y_train[best_idx]
best_point = X_train[best_idx]
latest_val = y_train[-1]

threshold = np.percentile(y_train, 100 - TOP_PERCENTILE)
n_pos = int(np.sum(y_train >= threshold))
n_cv_folds = max(2, min(5, int(min(n_pos, n_samples - n_pos) / 2)))

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

print('='*60)
print(f'  {FUNCTION_ID} Week {WEEK} -- DATA LOADED')
print('='*60)
print(f'  Samples      : {n_samples}')
print(f'  Dimensions   : {n_dims}D')
print(f'  n/p ratio    : {n_samples/n_dims:.2f}')
print(f'  Y range      : [{y_train.min():.6f}, {y_train.max():.6f}]')
print(f'  [ATB] BEST      : {best_value:.6f}  @ index {best_idx}')
print(f'   Location  : {best_point.round(4)}')
print(f'   Latest    : {latest_val:.6f}  (Week {WEEK-1})')
print(f'  Label threshold (top {TOP_PERCENTILE}%): {threshold:.6f}  ({n_pos} positives)')
print(f'  CV folds: {n_cv_folds}')

---
## 📊 STEP 3 — Historical Performance

### W1–W11 Submission History

| Week | Score | Note |
|------|-------|------|
| W1 | -2.6271 |  |
| W2 | 0.2376 | ATB |
| W3 | -0.962 |  |
| W4 | -0.5268 |  |
| W5 | -2.4571 |  |
| W6 | -0.1294 |  |
| W7 | -0.2651 |  |
| W8 | -0.5542 |  |
| W9 | -1.4047 |  |
| W10 | -1.8014 |  |
| W11 | 0.2376 | ATB |

| W12 | TBD | ATB OVERRIDE — replicate W2 ATB [0.439249, 0.414994, 0.384687, 0.397917] |

### Module 22 Clustering — ISOLATED ATTRACTOR

Top-3 spread=0.062. Single narrow positive basin — W2 and W11 (exact repeat) are the only positive results across 11 weeks. 9 surrounding attempts all negative. ATB→centroid dist=0.031. Move any dim >0.05 → fall back to negative.

**W12 decision from clustering:** W2 ATB is the only positive result in 11 weeks. W11 exact replication confirmed coords correct (matched 0.23759 exactly). GP sigma=2.25 (non-stationary landscape) — no trustworthy gradient signal to improve on this point.


In [ ]:
# Week-on-week bar chart
week_scores = [-2.6271, 0.2376, -0.962, -0.5268, -2.4571, -0.1294, -0.2651, -0.5542, -1.4047, -1.8014, 0.2376]
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#2ecc71' if v == max(week_scores) else '#e74c3c' if i > 0 and v < week_scores[i-1] else '#3498db' for i, v in enumerate(week_scores)]
ax.bar(range(1, 12), week_scores, color=colors, alpha=0.85, edgecolor='white')
ax.axhline(max(week_scores), color='gold', linestyle='--', linewidth=2, label=f'ATB: {max(week_scores):.4g}')
ax.set_xlabel('Week'); ax.set_ylabel('Output Value')
ax.set_title(f'F4 W1-W11 History + W12 Target (MAXIMIZE ⬆️)', fontweight='bold')
ax.set_xticks(range(1, 12)); ax.set_xticklabels([f'W{i}' for i in range(1, 12)])
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(f'{FUNCTION_ID}_W{WEEK}_Step3_History.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FUNCTION_ID}_W{WEEK}_Step3_History.png')

---
## 🏷️ STEP 4 — Binary Classification Labels

Top 30% = class 1 (HIGH / good). Classifier trained to distinguish high-value regions.


In [ ]:
y_labels = (y_train >= threshold).astype(int)
cv = StratifiedKFold(n_splits=n_cv_folds, shuffle=True, random_state=RANDOM_SEED)

models = {
    'Linear SVM'         : SVC(kernel='linear', C=1.0, probability=True, random_state=RANDOM_SEED),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    f'NN-Small {NN_SMALL}' : MLPClassifier(hidden_layer_sizes=NN_SMALL, max_iter=500, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED),
    f'NN-Medium {NN_MEDIUM}': MLPClassifier(hidden_layer_sizes=NN_MEDIUM, max_iter=500, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED),
    f'NN-Large {NN_LARGE}' : MLPClassifier(hidden_layer_sizes=NN_LARGE, max_iter=500, early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED),
    'CNN-1D (learning)'  : TinyCNNClassifier(n_epochs=200, lr=0.01, seed=RANDOM_SEED),
}

cv_results = {}
print(f'  {n_cv_folds}-FOLD CV -- {FUNCTION_ID} ({n_samples} samples, {n_dims}D)')
print(f'  {"Model":<28} {"Mean CV":>9} {"+-Std":>7}')
print('  '+'-'*48)

for name, model in models.items():
    try:
        scores = cross_val_score(model, X_scaled, y_labels, cv=cv, scoring='accuracy')
        cv_results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores, 'model': model}
        flag = '*' if scores.mean() == max(r['mean'] for r in cv_results.values()) else ' '
        print(f'  {flag} {name:<27} {scores.mean():>8.1%} {scores.std():>7.1%}')
    except Exception as e:
        cv_results[name] = {'mean': 0.0, 'std': 1.0, 'scores': np.zeros(n_cv_folds), 'model': model}
        print(f'  ✗ {name:<27} FAILED')

ranked_models = sorted(cv_results.items(), key=lambda x: x[1]['mean'], reverse=True)
best_name = ranked_models[0][0]
best_info = ranked_models[0][1]
print(f'\n  [OK] SELECTED: {best_name}  (CV={best_info["mean"]:.1%})')

trained_models = {}
for name, info in cv_results.items():
    try:
        info['model'].fit(X_scaled, y_labels)
        trained_models[name] = info['model']
    except: pass

---
## 🔬 STEP 5B — CNN Inspection (Module 17 Learning Exercise)

Conv1d(1→8, k=2) scans 3 adjacent coordinate pair(s). Filter weights and feature maps
at best-known point documented below.


In [ ]:
# CNN Module 17 inspection
cnn_model = trained_models.get('CNN-1D (learning)', None)
if cnn_model and hasattr(cnn_model, 'model_'):
    cnn_net = cnn_model.model_; cnn_net.eval()
    cnn_params = sum(p.numel() for p in cnn_net.parameters() if p.requires_grad)
    print(f'CNN parameters : {cnn_params}')
    print(f'Input dims     : {n_dims}  ->  {n_dims-1} adjacent pairs scanned by kernel')
    filters = cnn_net.conv.weight.data.cpu().numpy()[:,0,:]
    print(f'Filters shape  : {filters.shape}  (8 filters x kernel_size=2)')
    import torch as _t
    _x  = _t.FloatTensor(X_scaled[best_idx:best_idx+1]).unsqueeze(1)
    import torch.nn.functional as _F
    with _t.no_grad():
        _fm = _F.relu(cnn_net.conv(_x)).squeeze(0).numpy()
    peak_f = int(np.argmax(_fm.max(axis=1)))
    peak_p = int(np.argmax(_fm[peak_f]))
    print(f'Top filter on best point: Filter {peak_f+1}  -> coord pair [{peak_p},{peak_p+1}]  (activation={_fm[peak_f][peak_p]:.4f})')
else:
    print('CNN not available for inspection')

---
## 🎲 STEP 8 — Generate & Filter Candidates

**92% Exploitation** — Anisotropic Gaussian around best point with σ=[0.012, 0.012, 0.012, 0.012].

**7% Exploration** — Uniform random across [0,1]^4.

Winner classifier scores all 10,000 → bottom 50% discarded → remainder to GP.


In [ ]:
best_clf = trained_models[best_name]
n_exploit = int(N_CANDIDATES * EXPLOIT_RATIO)
n_explore = N_CANDIDATES - n_exploit

_aniso = np.array(ANISO_SIGMA)
X_exploit = np.clip(best_point + np.random.normal(0, 1, (n_exploit, n_dims)) * _aniso, 0, 1)
X_explore = np.random.uniform(0, 1, (n_explore, n_dims))
X_cands   = np.vstack([X_exploit, X_explore])
src_tag   = np.array(['exploit']*n_exploit + ['explore']*n_explore)

X_cands_sc = scaler.transform(X_cands)
proba_all  = best_clf.predict_proba(X_cands_sc)[:, 1]
keep_mask  = proba_all >= np.percentile(proba_all, FILTER_PERCENTILE)
X_filt     = X_cands[keep_mask]
proba_filt = proba_all[keep_mask]
src_filt   = src_tag[keep_mask]

print('='*60)
print('  CANDIDATE GENERATION & FILTERING')
print('='*60)
print(f'  Generated  : {N_CANDIDATES:,}  ({n_exploit:,} exploit + {n_explore:,} explore)')
print(f'  Filtered by: {best_name}')
print(f'  Kept       : {len(X_filt):,} (top {100-FILTER_PERCENTILE}% by P(class=1))')
print(f'  Exploit %  : {100*np.mean(src_filt=="exploit"):.1f}%')
print(f'  Prob range : [{proba_filt.min():.3f}, {proba_filt.max():.3f}]')

---
## 🔬 STEP 9 — Gaussian Process Regression

Matérn ν=5/2 kernel fitted on full W12 history. R²=1.0 expected (memorising) — GP used for
acquisition function computation, not extrapolation. ATB override takes precedence for final submission.


In [ ]:
kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True,
                               n_restarts_optimizer=GP_RESTARTS, random_state=RANDOM_SEED)
gp.fit(X_scaled, y_train)

y_pred_tr, _ = gp.predict(X_scaled, return_std=True)
ss_res = np.sum((y_train - y_pred_tr)**2)
ss_tot = np.sum((y_train - y_train.mean())**2)
r2   = 1 - ss_res/ss_tot if ss_tot > 0 else 0.0
rmse = np.sqrt(np.mean((y_train - y_pred_tr)**2))

X_filt_sc = scaler.transform(X_filt)
mu, sigma  = gp.predict(X_filt_sc, return_std=True)

print('='*60)
print('  GAUSSIAN PROCESS REGRESSION')
print('='*60)
print(f'  Fitted kernel: {gp.kernel_}')
print(f'  R2           : {r2:.4f}  {"[OK] Good" if r2>0.7 else "⚠️ Moderate" if r2>0.4 else "[FAIL] Poor"}')
print(f'  RMSE         : {rmse:.6f}')
print(f'  mu range      : [{mu.min():.4f}, {mu.max():.4f}]')
print(f'  sigma range      : [{sigma.min():.4f}, {sigma.max():.4f}]')

---
## 🎯 STEP 10 — Acquisition Functions: EI + UCB

EI = (μ − y_best − ξ)·Φ(Z) + σ·φ(Z).  UCB = μ + κ·σ.  Both computed across filtered candidates.


In [ ]:
imp     = mu - best_value - EI_XI
Z       = imp / (sigma + 1e-9)
EI_acq  = np.maximum(imp * norm.cdf(Z) + sigma * norm.pdf(Z), 0)
UCB_acq = mu + UCB_KAPPA * sigma

best_ei_idx = np.argmax(EI_acq)
submission  = X_filt[best_ei_idx]
sub_mu      = mu[best_ei_idx]
sub_sigma   = sigma[best_ei_idx]
sub_ei      = EI_acq[best_ei_idx]
sub_ucb     = UCB_acq[best_ei_idx]
exploit_term = imp[best_ei_idx] * norm.cdf(Z[best_ei_idx])
explore_term = sigma[best_ei_idx] * norm.pdf(Z[best_ei_idx])

print('='*60)
print('  ACQUISITION FUNCTIONS (MAXIMIZE ⬆️)')
print('='*60)
print(f'  EI max  : {EI_acq.max():.6f}   UCB max: {UCB_acq.max():.6f}')
print(f'  EI > 0  : {100*np.mean(EI_acq>0):.1f}% of candidates')
print(f'\n  * SELECTED (max EI):')
print(f'  Coords  : {submission}')
print(f'  mu       : {sub_mu:.6f}   sigma: {sub_sigma:.6f}')
print(f'  EI      : {sub_ei:.6f}   UCB: {sub_ucb:.6f}')
driver = 'EXPLOITATION' if exploit_term > explore_term else 'EXPLORATION'
print(f'  Driver  : {driver}')
status = '[OK] PREDICTS NEW BEST' if sub_mu > best_value else '[OK] PREDICTS RECOVERY' if sub_mu > latest_val else '🔍 EXPLORING'
print(f'  Status  : {status}  (mu={sub_mu:.4f} vs best={best_value:.4f})')

---
## 📈 STEP 11 — Per-Dimension Sensitivity

GP gradient at best point — which dimensions most influence μ.
Anisotropic sigma in Step 8 was set based on W11 sensitivity analysis.


In [ ]:
delta = 0.01
sensitivities = []
for d in range(n_dims):
    p_up = best_point.copy(); p_up[d] = min(1.0, best_point[d]+delta)
    p_dn = best_point.copy(); p_dn[d] = max(0.0, best_point[d]-delta)
    m_up, _ = gp.predict(scaler.transform(p_up.reshape(1,-1)), return_std=True)
    m_dn, _ = gp.predict(scaler.transform(p_dn.reshape(1,-1)), return_std=True)
    sensitivities.append(abs(float(m_up) - float(m_dn)) / (2*delta))

print('  PER-DIMENSION SENSITIVITY:')
print(f'  {"Dim":<6} {"Best val":>10} {"Sensitivity":>13}')
print('  '+'-'*32)
for d in range(n_dims):
    print(f'  X{d+1:<5} {best_point[d]:>10.4f} {sensitivities[d]:>13.4f}')
top2 = np.argsort(sensitivities)[::-1][:2]
print(f'\n  Top 2 sensitive dims: {", ".join([f"X{d+1} ({sensitivities[d]:.4f})" for d in top2])}')

---
## 🗺️ STEP 12 — GP Acquisition Surfaces

2D contour surfaces of μ, EI, UCB across the two most sensitive dimensions.
All other dimensions fixed at best-point values.


In [ ]:
delta = 0.01
sensitivities = []
for d in range(n_dims):
    p_up = best_point.copy(); p_up[d] = min(1.0, best_point[d]+delta)
    p_dn = best_point.copy(); p_dn[d] = max(0.0, best_point[d]-delta)
    m_up,_ = gp.predict(scaler.transform(p_up.reshape(1,-1)), return_std=True)
    m_dn,_ = gp.predict(scaler.transform(p_dn.reshape(1,-1)), return_std=True)
    sensitivities.append(abs(float(m_up)-float(m_dn))/(2*delta))
top2 = np.argsort(sensitivities)[::-1][:2]
d1, d2 = int(top2[0]), int(top2[1])
print(f'Top 2 sensitive dims: X{d1+1} ({sensitivities[d1]:.4f}), X{d2+1} ({sensitivities[d2]:.4f})')

g = 40; g1=np.linspace(0,1,g); g2=np.linspace(0,1,g)
G1,G2 = np.meshgrid(g1,g2)
gpts = np.tile(best_point,(g*g,1)); gpts[:,d1]=G1.ravel(); gpts[:,d2]=G2.ravel()
mg,sg = gp.predict(scaler.transform(gpts),return_std=True)
ig=mg-best_value-EI_XI; Zg=ig/(sg+1e-9)
EIg=np.maximum(ig*norm.cdf(Zg)+sg*norm.pdf(Zg),0)

fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,Zpl,ttl,cm_ in [(axes[0],mg.reshape(g,g),'GP Mean mu','viridis'),(axes[1],EIg.reshape(g,g),'Expected Improvement','plasma')]:
    cf=ax.contourf(G1,G2,Zpl,levels=20,cmap=cm_,alpha=0.88)
    ax.scatter(X_train[:,d1],X_train[:,d2],c='white',s=40,edgecolors='black',linewidths=0.8,zorder=4)
    ax.scatter(best_point[d1],best_point[d2],c='red',s=400,marker='*',zorder=6,edgecolors='white',linewidths=2,label='ATB')
    ax.set_xlabel(f'X{d1+1}'); ax.set_ylabel(f'X{d2+1}'); ax.set_title(ttl,fontweight='bold')
    ax.legend(); plt.colorbar(cf,ax=ax)
plt.suptitle(f'{FUNCTION_ID} W{WEEK} -- GP Surfaces (X{d1+1} x X{d2+1})',fontweight='bold')
plt.tight_layout(); plt.savefig(f'{FUNCTION_ID}_W{WEEK}_Step12_GPSurfaces.png',dpi=150,bbox_inches='tight')
plt.show()

---
## 🏆 STEP 13 — Submission Dashboard

Top-5 EI candidates + summary metrics.


In [ ]:
top5 = np.argsort(EI_acq)[::-1][:5]
print('='*65)
print('  TOP 5 CANDIDATES BY EI')
print('='*65)
print(f'  {"Rank":<6} {"EI":>10} {"GP mu":>10} {"GP sigma":>10}  Source')
print('  '+'-'*52)
for rank, idx in enumerate(top5,1):
    star = '*' if rank==1 else ' '
    print(f'  {star}#{rank:<4} {EI_acq[idx]:>10.6f} {mu[idx]:>10.4f} {sigma[idx]:>10.4f}  {src_filt[idx]}')
    print(f'          {X_filt[idx].round(4)}')

print()
print(f'  ATB        : {best_value:.6f}  (Week 2)')
print(f'  GP mu best : {sub_mu:.6f}')
print(f'  Delta      : {sub_mu - best_value:+.6f}')
print(f'  Override   : 0.439249-0.414994-0.384687-0.397917')

---
## 📋 STEP 14 — Final Formatted Submission

### ATB OVERRIDE ACTIVE

**Rationale:** Only positive result in 11 weeks. W11 exact match confirmed coords correct. Narrow positive basin.

**Submit this string:**

```
0.439249-0.414994-0.384687-0.397917
```

ATB score: 0.23759 (Week 2)


In [ ]:
# -- ATB OVERRIDE -- submit exact ATB coordinates -------------
# GP pipeline ran for documentation. Submission = ATB coords.
# Only positive result in 11 weeks. W11 exact match confirmed coords correct. Narrow positive basin.

ATB_OVERRIDE_STR = '0.439249-0.414994-0.384687-0.397917'
final_submission = np.array([float(x) for x in ATB_OVERRIDE_STR.split('-')])
submission_str   = ATB_OVERRIDE_STR

print('+' + '='*68 + '+')
print(f'|  WEEK {WEEK} FINAL SUBMISSION -- {FUNCTION_ID}{" "*38}|')
print('+' + '='*68 + '+')
print(f'|  ATB OVERRIDE ACTIVE{" "*47}|')
print(f'|  ATB score: {best_value:.6f}  (Week 2){" "*38}|')
print(f'|  Override rationale: Only positive result in 11 weeks. W11 exact {" "*0}|')
print('+' + '='*68 + '+')
print(f'|  COPY THIS TO SUBMIT:{" "*47}|')
print(f'|  {submission_str:<66}|')
print('+' + '='*68 + '+')

np.save(f'{FUNCTION_ID}_W{WEEK}_submission.npy', final_submission)
with open(f'{FUNCTION_ID}_W{WEEK}_submission.txt', 'w') as f:
    f.write(f'# {FUNCTION_ID} Week {WEEK} | ATB OVERRIDE | MAXIMIZE\n')
    f.write(f'# ATB: {best_value:.6f} @ W2 -- exact replication\n')
    f.write(f'# Only positive result in 11 weeks. W11 exact match confirmed coords correct. Narrow positive basin.\n')
    f.write(submission_str + '\n')
print(f'\n[OK] Saved: {FUNCTION_ID}_W{WEEK}_submission.txt')

---
## 💾 STEP 15 — Save Hyperparameter Record

Complete record of all settings, GP kernel, CV results, and submission string saved to JSON and TXT.


In [ ]:
import json, os
from datetime import datetime

record = {
    'function': FUNCTION_ID, 'week': WEEK,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'strategy': 'ATB OVERRIDE -- replicate W2 ATB [0.439249, 0.414994, 0.384687, 0.397917]',
    'cluster_type': 'ISOLATED ATTRACTOR',
    'atb_value': 0.23759, 'atb_week': 2,
    'override_str': ATB_OVERRIDE_STR,
    'hyperparameters': {
        'EXPLOIT_RATIO': EXPLOIT_RATIO, 'EXPLOIT_SIGMA': EXPLOIT_SIGMA,
        'ANISO_SIGMA': ANISO_SIGMA, 'EI_XI': EI_XI, 'UCB_KAPPA': UCB_KAPPA,
        'GP_RESTARTS': GP_RESTARTS, 'TOP_PERCENTILE': TOP_PERCENTILE,
        'FILTER_PERCENTILE': FILTER_PERCENTILE,
    },
    'gp_kernel': str(gp.kernel_), 'gp_r2': round(float(r2),6), 'gp_rmse': round(float(rmse),8),
    'best_model': best_name, 'best_model_cv': round(float(best_info['mean']),4),
    'submission_string': ATB_OVERRIDE_STR,
}

fname = f'{FUNCTION_ID}_W{WEEK}_hyperparameters.json'
with open(fname,'w') as f: json.dump(record, f, indent=2)
print(f'[OK] Saved: {fname}')

with open(f'{FUNCTION_ID}_W{WEEK}_run_report.txt','w') as f:
    f.write(f'BBO Run Report -- {FUNCTION_ID} Week {WEEK}\n')
    f.write('='*60+'\n')
    f.write(f'Strategy  : ATB OVERRIDE -- replicate W2 ATB [0.439249, 0.414994, 0.384687, 0.397917]\n')
    f.write(f'Clustering: ISOLATED ATTRACTOR\n')
    f.write(f'ATB       : {best_value:.6f} @ W2\n')
    f.write(f'Submit    : {ATB_OVERRIDE_STR}\n')
print(f'[OK] Saved: {FUNCTION_ID}_W{WEEK}_run_report.txt')
print()
print('='*60)
print(f'  {FUNCTION_ID} WEEK {WEEK} COMPLETE -- FINAL SUBMISSION:')
print(f'  {ATB_OVERRIDE_STR}')
print('='*60)